# Phase 6: Export Inference Artifact for the FastAPI Worker

Package the trained model + all state the worker needs to score a single incoming transaction.
Serialize everything into a single fraud_artifact.joblib file

In [1]:
import json
import numpy as np
import pandas as pd
import joblib
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder

TABULAR_PATH  = "/Users/shengqu/Desktop/CSCI5253/FinalProject/ieee-fraud-detection/data/train_with_uid.parquet"
GRAPH_PATH    = "/Users/shengqu/Desktop/CSCI5253/FinalProject/ieee-fraud-detection/data/graph_features.parquet"
MODEL_PATH    = "/Users/shengqu/Desktop/CSCI5253/FinalProject/model_output/xgb_combined.json"
ARTIFACT_PATH = "/Users/shengqu/Desktop/CSCI5253/FinalProject/model_output/fraud_artifact.joblib"

TRAIN_FRACTION  = 0.80
SMOOTHING_ALPHA = 10

## 1. Load data & model

In [2]:
df = pd.read_parquet(TABULAR_PATH).sort_values("TransactionDT").reset_index(drop=True)
graph = pd.read_parquet(GRAPH_PATH)
df = df.merge(graph, on="TransactionID", how="left")

split_idx = int(len(df) * TRAIN_FRACTION)
train_df = df.iloc[:split_idx]
global_rate = float(train_df["isFraud"].mean())

model = xgb.XGBClassifier()
model.load_model(MODEL_PATH)
print(f"Loaded model with {len(model.feature_names_in_)} features.")

Loaded model with 673 features.


## 2. Rebuild graph state as plain Python dicts

In [ ]:
def to_map(series):
    return {k: float(v) for k, v in series.dropna().items()}

# Neighbor fraud rate 
def smoothed_fraud_rate(train_slice, entity_col):
    g = train_slice.groupby(entity_col)["isFraud"].agg(["sum", "count"])
    g["rate"] = (g["sum"] + SMOOTHING_ALPHA * global_rate) / (g["count"] + SMOOTHING_ALPHA)
    return g["rate"]

nbr_entities = ["card1", "P_emaildomain", "R_emaildomain",
                "addr1", "DeviceInfo", "id_31"]
nbr_fraud_rate = {e: to_map(smoothed_fraud_rate(train_df, e)) for e in nbr_entities}
print("Neighbor fraud-rate tables built for:", list(nbr_fraud_rate.keys()))

Neighbor fraud-rate tables built for: ['card1', 'P_emaildomain', 'R_emaildomain', 'addr1', 'DeviceInfo', 'id_31']


In [ ]:
# Degree 
pairs = [
    ("uid", "card1"), ("uid", "P_emaildomain"), ("uid", "DeviceInfo"), ("uid", "addr1"),
    ("card1", "uid"), ("card1", "P_emaildomain"), ("card1", "addr1"),
    ("P_emaildomain", "uid"), ("P_emaildomain", "card1"),
    ("addr1", "uid"), ("addr1", "card1"),
    ("DeviceInfo", "uid"),
]

degree = {}
for e, n in pairs:
    series = df.groupby(e)[n].nunique()
    degree[f"{e}__nunique_{n}"] = {k: int(v) for k, v in series.items()}
print(f"Degree tables: {len(degree)}")

Degree tables: 12


In [ ]:
# Amount aggregates
amt_stats = {}
for e in ["uid", "card1", "P_emaildomain", "addr1"]:
    g = df.groupby(e)["TransactionAmt"].agg(["mean", "std"])
    amt_stats[e] = {k: (float(r["mean"]), float(r["std"]) if pd.notna(r["std"]) else 0.0)
                    for k, r in g.iterrows()}
print(f"Amount stats for: {list(amt_stats.keys())}")

Amount stats for: ['uid', 'card1', 'P_emaildomain', 'addr1']


## 3. Preprocessing metadata

In [ ]:
# Reconstruct the exact train/valid split used in Phase 5 
drop_cols = ["isFraud", "TransactionID", "TransactionDT", "uid", "card_age_day", "is_train"]
drop_cols = [c for c in drop_cols if c in df.columns]
X_full = df.drop(columns=drop_cols).copy()
X_train_ref = X_full.iloc[:split_idx]

# High-missingness columns to drop (computed from train only)
miss = X_train_ref.isnull().mean()
drop_high_miss = miss[miss > 0.90].index.tolist()
X_full.drop(columns=drop_high_miss, inplace=True)
X_train_ref = X_full.iloc[:split_idx]
print(f"Columns to drop at inference ({len(drop_high_miss)}): {drop_high_miss}")

# Mid-missingness columns that need _missing indicators
miss = X_train_ref.isnull().mean()
missing_indicator_cols = miss[(miss > 0.50) & (miss <= 0.90)].index.tolist()
print(f"Columns needing _missing indicator: {len(missing_indicator_cols)}")

Columns to drop at inference (12): ['dist2', 'D7', 'id_07', 'id_08', 'id_18', 'id_21', 'id_22', 'id_23', 'id_24', 'id_25', 'id_26', 'id_27']
Columns needing _missing indicator: 218


In [ ]:
# Label encoders
for c in missing_indicator_cols:
    X_full[f"{c}_missing"] = X_full[c].isnull().astype(np.int8)

label_encoders = {}
for c in X_full.select_dtypes(include="object").columns:
    le = LabelEncoder()
    values = X_full[c].astype(str).fillna("NA")
    le.fit(values)
    label_encoders[c] = {v: int(code) for v, code in zip(le.classes_, le.transform(le.classes_))}

print(f"Label encoders saved for {len(label_encoders)} columns.")

/var/folders/ws/c4n8v3rs1_g18bkf6s3m1lhc0000gn/T/ipykernel_87680/3089125038.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_full[f"{c}_missing"] = X_full[c].isnull().astype(np.int8)
/var/folders/ws/c4n8v3rs1_g18bkf6s3m1lhc0000gn/T/ipykernel_87680/3089125038.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_full[f"{c}_missing"] = X_full[c].isnull().astype(np.int8)
/var/folders/ws/c4n8v3rs1_g18bkf6s3m1lhc0000gn/T/ipykernel_87680/3089125038.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the res

Label encoders saved for 29 columns.


## 4. Feature order


In [8]:
feature_order = list(model.feature_names_in_)
print(f"{len(feature_order)} features expected by the model.")
print("First 5:", feature_order[:5])
print("Last 5: ", feature_order[-5:])

673 features expected by the model.
First 5: [np.str_('TransactionAmt'), np.str_('ProductCD'), np.str_('card1'), np.str_('card2'), np.str_('card3')]
Last 5:  [np.str_('id_37_missing'), np.str_('id_38_missing'), np.str_('DeviceType_missing'), np.str_('DeviceInfo_missing'), np.str_('DeviceInfo__nunique_uid_missing')]


## 5. Velocity state snapshot

In [9]:
WINDOW_SEC = 7 * 86400
cutoff = train_df["TransactionDT"].max() - WINDOW_SEC
recent = train_df[train_df["TransactionDT"] >= cutoff][[
    "TransactionID", "TransactionDT", "uid", "card1"
]]

velocity_snapshot = {}
for e in ["uid", "card1"]:
    velocity_snapshot[e] = (
        recent.groupby(e)["TransactionDT"]
              .apply(lambda x: sorted(int(t) for t in x))
              .to_dict()
    )
print(f"Velocity snapshot: {len(velocity_snapshot['uid']):,} uids, "
      f"{len(velocity_snapshot['card1']):,} card1s over the last 7 days of training.")

Velocity snapshot: 14,110 uids, 3,049 card1s over the last 7 days of training.


## 6. UID construction parameters


In [10]:
uid_spec = {
    "fields": ["card1", "addr1", "P_emaildomain"],
    "derive_card_age_day": {
        "formula": "floor(TransactionDT / 86400) - D1",
        "missing_D1_sentinel": -1
    },
    "template": "{card1}_{addr1}_{P_emaildomain}_{card_age_day}"
}
print(json.dumps(uid_spec, indent=2))

{
  "fields": [
    "card1",
    "addr1",
    "P_emaildomain"
  ],
  "derive_card_age_day": {
    "formula": "floor(TransactionDT / 86400) - D1",
    "missing_D1_sentinel": -1
  },
  "template": "{card1}_{addr1}_{P_emaildomain}_{card_age_day}"
}


## 7. Bundle and save

In [11]:
artifact = {
    "model": model,
    "feature_order": feature_order,
    "preprocessing": {
        "drop_high_miss": drop_high_miss,
        "missing_indicator_cols": missing_indicator_cols,
        "label_encoders": label_encoders,
    },
    "graph_state": {
        "nbr_fraud_rate": nbr_fraud_rate,
        "degree": degree,
        "amt_stats": amt_stats,
        "velocity_snapshot": velocity_snapshot,
        "global_rate": global_rate,
    },
    "uid_spec": uid_spec,
    "metadata": {
        "smoothing_alpha": SMOOTHING_ALPHA,
        "train_fraction": TRAIN_FRACTION,
        "n_train_rows": int(split_idx),
    }
}

joblib.dump(artifact, ARTIFACT_PATH, compress=3)
import os
size_mb = os.path.getsize(ARTIFACT_PATH) / 1024**2
print(f"Saved {ARTIFACT_PATH}  ({size_mb:.1f} MB)")

Saved /Users/shengqu/Desktop/CSCI5253/FinalProject/model_output/fraud_artifact.joblib  (5.7 MB)


## 8. Test — scoring a single raw transaction

In [ ]:
art = joblib.load(ARTIFACT_PATH)

def build_uid(row, spec):
    day = int(row["TransactionDT"] // 86400)
    d1  = row.get("D1")
    d1  = spec["derive_card_age_day"]["missing_D1_sentinel"] if pd.isna(d1) else int(d1)
    card_age_day = day - d1
    parts = {f: str(row.get(f, "NA") or "NA") for f in spec["fields"]}
    parts["card_age_day"] = str(card_age_day)
    return spec["template"].format(**parts)

def attach_graph_features(row, art):
    gs = art["graph_state"]
    out = {}
    # neighbor fraud rate 
    for e, table in gs["nbr_fraud_rate"].items():
        key = row.get(e)
        out[f"{e}__nbr_fraud_rate"] = table.get(key, gs["global_rate"])
    # degree 
    for col, table in gs["degree"].items():
        entity_col = col.split("__")[0]
        out[col] = table.get(row.get(entity_col), 1)
    # amount stats
    for e, table in gs["amt_stats"].items():
        mean, std = table.get(row.get(e), (row["TransactionAmt"], 0.0))
        out[f"{e}__amt_mean"]  = mean
        out[f"{e}__amt_std"]   = std
        out[f"{e}__amt_ratio"] = row["TransactionAmt"] / (mean + 1e-6)
    # velocity 
    for e in ["uid", "card1"]:
        for label in ["1h", "24h", "7d"]:
            out[f"{e}__vel_{label}"] = 0
    return out

def preprocess(row, art):
    pp = art["preprocessing"]
    row = dict(row)
    # attach graph features
    row.update(attach_graph_features(row, art))
    # missingness indicators
    for c in pp["missing_indicator_cols"]:
        row[f"{c}_missing"] = int(pd.isna(row.get(c)))
    # label encode
    for c, mapping in pp["label_encoders"].items():
        v = row.get(c)
        v = "NA" if pd.isna(v) else str(v)
        row[c] = mapping.get(v, 0)   
    # assemble vector in the exact feature order
    vec = [row.get(f, np.nan) for f in art["feature_order"]]
    return np.array(vec, dtype=float).reshape(1, -1)

def score(raw_row, art):
    row = dict(raw_row)
    row["uid"] = build_uid(row, art["uid_spec"])
    X = preprocess(row, art)
    return float(art["model"].predict_proba(X)[:, 1][0])

# Take one row from the validation slice and score it
sample = df.iloc[split_idx + 1000].to_dict()
p = score(sample, art)
print(f"Fraud probability for sample transaction: {p:.4f}")
print(f"True label: {int(sample['isFraud'])}")

Fraud probability for sample transaction: 0.0100
True label: 0
